In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc PyGithub networkx qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Iskay Quantum Optimizer - Uma Função Qiskit da Kipu Quantum

> **Note:** * As Qiskit Functions são um recurso experimental disponível apenas para usuários dos planos IBM Quantum&reg; Premium Plan, Flex Plan e On-Prem (via IBM Quantum Platform API). Elas estão em status de lançamento de pré-visualização e sujeitas a alterações.

## Visão geral
Com o Iskay Quantum Optimizer da Kipu Quantum, você pode resolver problemas de otimização complexos utilizando computadores quânticos IBM&reg;. Este solver utiliza o avançado [algoritmo bf-DCQO](https://doi.org/10.48550/arXiv.2409.04477) da Kipu, que requer apenas a função objetivo como entrada para entregar automaticamente as soluções dos problemas. Ele consegue lidar com problemas de otimização envolvendo até 156 qubits, permitindo o uso de todos os qubits dos dispositivos quânticos IBM. O Optimizer usa um mapeamento 1-para-1 entre variáveis clássicas e qubits, o que permite resolver problemas de otimização com até 156 variáveis binárias.

O Optimizer permite resolver problemas de otimização binária sem restrições. Além da formulação QUBO (Quadratic Unconstrained Binary Optimization) comumente utilizada, ele também suporta problemas de otimização de ordem superior (HUBO). O solver utiliza um algoritmo quântico não-variacional, executando a maior parte do cálculo em dispositivos quânticos.

A seguir, há mais detalhes sobre o algoritmo utilizado, um breve guia de como usar a função e resultados de benchmarking em diversas instâncias de problemas de diferentes tamanhos e complexidades.

## Descrição
O Optimizer é uma implementação pronta para uso de algoritmos de otimização quântica de ponta. Ele resolve problemas de otimização executando circuitos quânticos altamente comprimidos em hardware quântico. Essa compressão é obtida introduzindo termos contradiabáticos na evolução temporal subjacente do sistema quântico. O algoritmo executa várias iterações de execuções no hardware para obter as soluções finais e as combina com pós-processamento. Essas etapas são integradas de forma transparente no fluxo de trabalho do Optimizer e são executadas automaticamente.

### Como o Quantum Optimizer funciona?
Esta seção descreve os fundamentos do algoritmo bf-DCQO implementado. Uma introdução ao algoritmo também pode ser encontrada no [canal do YouTube do Qiskit.](https://www.youtube.com/watch?v=33QmsXhIlpU&t=1223s)

O algoritmo é baseado na evolução temporal de um sistema quântico que se transforma ao longo do tempo, onde a solução do problema é codificada no estado fundamental do sistema quântico ao final da evolução. De acordo com o [teorema adiabático](https://en.wikipedia.org/wiki/Adiabatic_theorem), essa evolução deve ser lenta para garantir que o sistema permaneça em seu estado fundamental. Digitalizar essa evolução é a base da computação adiabática quântica digitalizada (DQA) e do famoso algoritmo QAOA. No entanto, a evolução lenta necessária não é viável para problemas de tamanho crescente, pois resulta em um aumento da profundidade do circuito. Ao usar protocolos contradiabáticos, é possível suprimir excitações indesejadas que ocorrem durante tempos de evolução curtos, permanecendo no estado fundamental. Aqui, digitalizar esse tempo de evolução mais curto resulta em circuitos quânticos com menor profundidade e menos portas de emaranhamento.

Os circuitos dos algoritmos bf-DCQO geralmente usam até dez vezes menos portas de emaranhamento do que o DQA, e três a quatro vezes menos portas de emaranhamento do que implementações padrão do QAOA. Por causa do menor número de portas, ocorrem menos erros durante a execução do circuito no hardware. Portanto, o optimizer não requer o uso de técnicas como supressão de erros ou mitigação de erros. Implementá-las em versões futuras pode melhorar ainda mais a qualidade da solução.

Embora o algoritmo bf-DCQO use iterações, ele é não-variacional. Após cada iteração do algoritmo, a distribuição de estados é medida. A distribuição obtida é usada para calcular um chamado campo de polarização (bias-field). O campo de polarização permite iniciar a próxima iteração a partir de um estado de energia próximo à solução encontrada anteriormente. Dessa forma, o algoritmo se move a cada iteração para soluções de menor energia. Tipicamente, aproximadamente dez iterações são suficientes para convergir para uma solução, exigindo no total um número muito menor de iterações do que os algoritmos variacionais, que é da ordem de aproximadamente 100 iterações.

O optimizer combina o algoritmo bf-DCQO com pós-processamento clássico. Após medir a distribuição de estados, uma busca local é realizada. Durante a busca local, os bits da solução medida são invertidos aleatoriamente. Após a inversão, a energia da nova string de bits é avaliada. Se a energia for menor, a string de bits é mantida como a nova solução. A busca local escala apenas linearmente com o número de qubits; portanto, é computacionalmente barata. Uma vez que o pós-processamento corrige inversões de bits locais, ele compensa erros de inversão de bit que frequentemente são resultado de imperfeições do hardware e erros de leitura.

### Fluxo de trabalho
A seguir, um esquema do fluxo de trabalho do Quantum Optimizer.

![Fluxo de trabalho](../docs/images/guides/kipu-optimization/workflow.svg "Fluxo de trabalho do Quantum Optimizer")

Ao usar o Quantum Optimizer, resolver um problema de otimização em hardware quântico pode ser reduzido a:
* Formular a função objetivo do problema
* Acessar o Optimizer via Qiskit Functions
* Executar o Optimizer e coletar o resultado

## Benchmarks
As métricas de benchmark abaixo mostram que o Optimizer aborda efetivamente problemas envolvendo até 156 qubits e oferecem uma visão geral da precisão e escalabilidade do optimizer em diferentes tipos de problemas. Note que as métricas de desempenho reais podem variar dependendo das características específicas do problema, como o número de variáveis, a densidade e localidade dos termos na função objetivo e a ordem polinomial.

A tabela a seguir inclui a razão de aproximação (AR), uma métrica definida da seguinte forma:
$$
AR = \frac{C^{*} - C_\textrm{max}}{C_{\textrm{min}} - C_{\textrm{max}}},
$$
onde $C$ é a função objetivo, $C_{\textrm{min}}$, $C_{\textrm{max}}$ são seus valores mínimo e máximo, e $C^{*}$ é o custo da melhor solução encontrada, respectivamente. Portanto, AR=100\% significa que o estado fundamental do problema foi obtido.

| Exemplo           | Número de qubits | Razão de Aproximação | Tempo total (s) | Uso de runtime (s) | Número total de shots | Número de iterações |
| ----------------- | :--------------: | :------: | :------------: | :---------------: | :-------------------: | :------------------: |
| Unweighted MaxCut |        28        |   100%   |      180       |        30         |          30k          |          5           |
| Unweighted MaxCut |        30        |   100%   |      180       |        30         |          30k          |          5           |
| Unweighted MaxCut |        32        |   100%   |      180       |        30         |          30k          |          5           |
| Unweighted MaxCut |        80        |   100%   |      480       |        60         |          90k          |          9           |
| Unweighted MaxCut |       100        |   100%   |      330       |        60         |          60k          |          6           |
| Unweighted MaxCut |       120        |   100%   |      370       |        60         |          60k          |          6           |
| HUBO 1            |       156        |   100%   |      600       |        70         |         100k          |          10          |
| HUBO 2            |       156        |   100%   |      600       |        70         |         100k          |          10          |

- As instâncias MaxCut com 28, 30 e 32 qubits foram executadas no ibm_sherbrooke. As instâncias com 80, 100 e 120 foram executadas em um processador Heron r2.
- As instâncias HUBO também foram executadas em um processador Heron r2.

Todas as instâncias de benchmark estão acessíveis no GitHub (veja [instâncias de benchmark da Kipu](https://github.com/Kipu-Quantum-GmbH/benchmark-instances)). Um exemplo para executar essas instâncias pode ser encontrado em [Exemplo 3: Instâncias de benchmark](#example-3-benchmark-instances).

## Entradas e saídas
### Entrada
Veja a tabela a seguir para todos os parâmetros de entrada que o Quantum Optimizer aceita. A seção de **Opções** subsequente fornece mais detalhes sobre as `options` disponíveis.

| Nome         | Tipo               | Descrição                                                                                                                                                                            | Obrigatório | Padrão                                                                                       | Exemplo                                                                                                              |
| ------------ | ------------------ | :------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | -------- | --------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------- |
| problem      | `Dict[str, float]` | Os coeficientes do problema de otimização formulado no formato QUBO/HUBO ou spin. Para mais informações sobre a especificação do problema, veja [Formatos de problema aceitos](#accepted-problem-formats)                                       | Sim      | N/A                                                                                           | `{"()": -21.0, "(0, 4)": 0.5,"(0, 2)": 0.5,"(0, 1)": 0.5,"(1, 3)": 0.5}`                                             |
| problem_type | `str`              | Especifique se os coeficientes do problema estão no formato binário (QUBO/HUBO) ou spin. As duas possibilidades são `"spin"` ou `"binary"`                                                   | Sim      | N/A                                                                                           | `"spin"`                                                                                                             |
| backend_name | `str`              | Nome do backend para realizar a consulta                                                                                                                                                  | Sim       | N/A                                                      | `"ibm_fez"`                                                                                                          |                                                                                         |
| options      | `Dict[str, Any]`   | Opções para gerenciar o envio ao hardware, como número de shots. Para mais detalhes sobre a configuração de opções, veja a seção de [Opções](#options) | Não       | Para ver os valores padrão da configuração de opções, veja a seção de [Opções](#options)                                                         | `{"shots": 5000, "num_iterations": 3, "use_session": True, "seed_transpiler": 42}` |

#### Formatos de problema aceitos
Os argumentos `problem` e `problem_type` codificam um problema de otimização da forma

$$
\begin{align}
\min_{(x_1, x_2, \ldots, x_n) \in D} C(x_1, x_2, \ldots, x_n) \nonumber
\end{align}
$$
onde

$$
C(x_1, ... , x_n) = a + \sum_{i} b_i x_i + \sum_{i, j} c_{i, j} x_i x_j + ... + \sum_{k_1, ..., k_m} g_{k_1, ..., k_m} x_{k_1} ... x_{k_m}
$$

- Ao escolher `problem_type = "binary"`, você especifica que a função de custo está no formato `binary`, o que significa que $D = {0,  1}^{n}$, ou seja, a função de custo está escrita na formulação QUBO/HUBO.
- Por outro lado, ao escolher `problem_type = "spin"`, a função de custo é escrita na formulação de Ising, onde $D = {-1, 1}^{n}$.

Os coeficientes do problema devem ser codificados em um dicionário da seguinte forma:
$$
\begin{align} \nonumber
&\texttt{{} \\ \nonumber
&\texttt{"()"}&: \quad &a, \\ \nonumber
&\texttt{"(i,)"}&: \quad &b_i, \\ \nonumber
&\texttt{"(i, j)"}&: \quad &c_{i, j}, \\ \nonumber
&\quad  \vdots \\ \nonumber
&\texttt{"(} k_1, ..., k_m  \texttt{)"} &: \quad &g_{k_1, ..., k_m}, \\ \nonumber
&\texttt{}}
\end{align}
$$

- Note que as chaves do dicionário devem ser strings contendo uma tupla válida de inteiros não repetidos.

#### Opções
O Iskay fornece capacidades de ajuste fino por meio de parâmetros opcionais. Embora os padrões funcionem bem para a maioria dos problemas, você pode personalizar o comportamento para requisitos específicos:

| Parâmetro | Tipo | Padrão | Descrição |
|-----------|------|---------|-------------|
| **shots** | `int` | 10000 | Medições quânticas por iteração (mais alto = mais preciso) |
| **num_iterations** | `int` | 10 | Iterações do algoritmo (mais iterações podem melhorar a qualidade da solução) |
| **use_session** | `bool` | True | Usar sessões IBM para reduzir tempos de fila |
| **seed_transpiler** | `int` | None | Definir para compilação reprodutível de Circuit quântico |
| **direct_qubit_mapping** | `bool` | False | Mapear qubits virtuais diretamente para qubits físicos |
| **job_tags** | `List[str]` | None | Tags personalizadas para rastreamento de jobs |
| **preprocessing_level** | `int` | 0 | Intensidade de pré-processamento do problema (0-3) - veja detalhes abaixo |
| **postprocessing_level** | `int` | 2 | Nível de refinamento da solução (0-2) - veja detalhes abaixo |
| **transpilation_level** | `int` | 0 | Tentativas de otimização do Transpiler (0-5) - veja detalhes abaixo |
| **transpile_only** | `bool` | False | Analisar a otimização do circuito sem executar a execução completa |

**Níveis de Pré-processamento (0-3)**: Especialmente importantes para problemas maiores que atualmente não cabem nos tempos de coerência do hardware. Níveis de pré-processamento mais altos alcançam profundidades de circuito menores por meio de aproximações na transpilação do problema:
- **Nível 0**: Exato, circuitos mais longos
- **Nível 1**: Bom equilíbrio entre precisão e aproximação, eliminando apenas as portas com ângulos no percentil mais baixo de 10%
- **Nível 2**: Aproximação ligeiramente maior, eliminando as portas com ângulos no percentil mais baixo de 20% e usando `approximation_degree=0.95` na transpilação
- **Nível 3**: Nível máximo de aproximação, eliminando as portas no percentil mais baixo de 30% e usando `approximation_degree=0.90` na transpilação

**Níveis de Transpilação (0-5)**: Controlam as tentativas avançadas de otimização do transpiler para compilação de Circuit quântico. Isso pode levar a um aumento na sobrecarga clássica e, em alguns casos, pode não alterar a profundidade do circuito. O valor padrão `2` geralmente leva ao menor circuito e é relativamente rápido.
- **Nível 0**: Otimização do circuito DCQO decomposto (layout, roteamento, agendamento)
- **Nível 1**: Otimização de `PauliEvolutionGate` e depois do circuito DCQO decomposto (max_trials=10)
- **Nível 2**: Otimização de `PauliEvolutionGate` e depois do circuito DCQO decomposto (max_trials=15)
- **Nível 3**: Otimização de `PauliEvolutionGate` e depois do circuito DCQO decomposto (max_trials=20)
- **Nível 4**: Otimização de `PauliEvolutionGate` e depois do circuito DCQO decomposto (max_trials=25)
- **Nível 5**: Otimização de `PauliEvolutionGate` e depois do circuito DCQO decomposto (max_trials=50)

**Níveis de Pós-processamento (0-2)**: Controlam quanto de otimização clássica, compensando erros de inversão de bit com diferentes números de passagens gulosas de uma busca local:
- **Nível 0**: 1 passagem
- **Nível 1**: 2 passagens
- **Nível 2**: 3 passagens

**Modo somente transpilação**: Agora disponível para usuários que desejam analisar a otimização de circuitos sem executar a execução completa do algoritmo quântico.

**Exemplo de configuração personalizada**: Veja como você pode configurar o Iskay com diferentes configurações:

In [1]:
custom_options = {
    "shots": 15_000,  # Higher shot count for better statistics
    "num_iterations": 12,  # More iterations for solution refinement
    "preprocessing_level": 1,  # Light preprocessing for problem simplification
    "postprocessing_level": 2,  # Maximum postprocessing for solution quality
    "transpilation_level": 3,  # Using higher transpilation level for circuit optimization
    "seed_transpiler": 42,  # Fixed seed for reproducible results
    "job_tags": ["custom_config"],  # Custom tracking tags
}

**Otimização de seed**: Note que `seed_transpiler` é definido como `None` por padrão. Isso habilita o processo de otimização automática do transpiler. Quando `None`, o sistema iniciará uma tentativa com múltiplos seeds e selecionará aquele que produz a melhor profundidade de circuito, aproveitando ao máximo o parâmetro `max_trials` para cada nível de transpilação.

**Desempenho do nível de transpilação**: Aumentar o número de `max_trials` com valores mais altos para `transpilation_level` inevitavelmente aumentará o tempo de transpilação, mas pode não alterar sempre o circuito final — isso depende muito da estrutura e complexidade específicas do circuito. Para alguns circuitos/problemas, no entanto, a diferença entre 10 tentativas (nível 1) e 50 tentativas (nível 5) pode ser dramática, então explorar esses parâmetros pode ser a chave para encontrar uma solução com sucesso.

### Saída
| Nome   | Tipo               | Descrição                                                           | Exemplo                                                                |
| ------ | ------------------ | ----------------------------------------------------------------------| ---------------------------------------------------------------------- |
| result | `Dict[str, Any]` | Solução e metadados. A estrutura varia com base na opção `transpile_only`. |   Veja "Conteúdo do dicionário de resultado" abaixo      |

### Conteúdo do dicionário de resultado
A estrutura do dicionário de resultado depende do modo de execução:

| Campo         | Tipo               | Modo | Descrição                                                                                                             | Exemplo |
| ------------ | :------------------: | :--: | ----------------------------------------------------------------------------------------------------------------------- | ------- |
| **solution**     | `Dict[str, int]` | **Padrão** | A solução mapeada ordenada onde as chaves são índices de variáveis (como strings) ordenados numericamente e os valores são os valores correspondentes das variáveis (1/-1 para problemas spin, 1/0 para problemas binários).| `{'0': -1, '1': -1, '2': -1, '3': 1, '4': 1}` |
| **solution_info**      | `Dict[str, Any]`   | **Padrão** | Informações detalhadas sobre a solução (veja detalhes abaixo)                                           | `{'bitstring': '11100', 'cost': -13.8, 'seed_transpiler': 42, 'mapping': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}}` |
| **prob_type** | `str`             | **Padrão** | O tipo do problema de otimização ('spin' ou 'binary')                                                           | `'spin'` |
| **transpilation_info** | `Dict[str, Any]` | **Somente transpilação** | Análise do circuito e detalhes de transpilação (veja detalhes abaixo) | `{'best_seed': 42, 'transpilation_time_seconds': 50.06, 'transpiled_circuit': {'depth': 576, 'gate_count': 4177, 'num_qubits': 156, 'width': 176, 'operations': {'sx': 1325, 'rx': 891, 'cz': 783, 'rz': 650, 'rzz': 466, 'x': 42, 'measure': 20}}}` |

#### Execução padrão
Quando o parâmetro opcional `transpile_only=False`:

**Dicionário solution_info:**
- **"bitstring"** (`str`): A representação em string de bits bruta da solução.
- **"cost"** (`float`): O valor de custo/energia associado à solução.
- **"seed_transpiler"** (`int`): A semente aleatória usada para o transpiler que produziu este resultado.
- **"mapping"** (`Dict[int, int]`): O mapeamento original qubit-para-variável usado na computação.
- **"qpu_time"** (`float`, opcional): O tempo de execução na QPU em segundos.

**Notas sobre mapeamento de variáveis:**
- O dicionário `solution` é obtido a partir da string de bits da solução, usando o objeto `mapping` para indexar as variáveis.
- Quando `problem_type=spin`, usamos a atribuição $1 \rightarrow -1, \quad 0 \rightarrow 1$.
- As chaves no dicionário de solução são índices de variáveis ordenados numericamente como strings.

#### Análise de transpilação
Quando o parâmetro opcional `transpile_only=True`:

**Dicionário transpilation_info:**
- **"best_seed"** (`int`): A seed ótima encontrada para transpilação
- **"transpilation_time_seconds"** (`float`): Tempo gasto no processo de transpilação
- **"transpiled_circuit"** (`Dict`): Análise do circuito contendo:
  - **"depth"** (`int`): Profundidade do circuito (número de camadas)
  - **"gate_count"** (`int`): Número total de portas no circuito
  - **"num_qubits"** (`int`): Número de qubits utilizados
  - **"width"** (`int`): Largura do circuito
  - **"operations"** (`Dict[str, int]`): Contagem de cada tipo de porta utilizada

**Uso do modo somente transpilação:**
- Disponível para usuários que desejam analisar a otimização de circuitos sem executar a execução completa do algoritmo quântico.
- Útil para análise de circuitos, estudos de otimização de profundidade e compreensão dos efeitos de transpilação antes de se comprometer com a execução completa.

## Primeiros passos
Nesta documentação, percorreremos as etapas de uso do Iskay Quantum Optimizer. No processo, mostraremos rapidamente como carregar a função do catálogo e como converter seu problema em uma entrada válida, ao mesmo tempo em que mostramos como você pode experimentar diferentes parâmetros opcionais.

Para um exemplo mais detalhado, consulte o tutorial [Resolva o problema de Market Split com o Iskay Quantum Optimizer da Kipu Quantum](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer), onde trabalhamos todo o processo de uso do Iskay Solver para resolver o problema de Market Split, que representa um desafio real de alocação de recursos onde os mercados devem ser particionados em regiões de vendas equilibradas para atender às metas exatas de demanda.

Autentique-se usando sua chave de API, encontrada no [painel do IBM Quantum Platform](http://quantum.cloud.ibm.com/), e selecione a Qiskit Function da seguinte forma:

In [ ]:
# ruff: noqa: F821

> **Note:** O código a seguir pressupõe que você salvou suas credenciais. Caso não tenha feito isso, siga as instruções em [salvar sua conta IBM Cloud](/guides/functions#install-qiskit-functions-catalog-client) para autenticar com sua chave de API.

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(
    channel="ibm_quantum_platform",
    instance="INSTANCE_CRN",
    token="YOUR_API_KEY",  # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
)

# Access Function
optimizer = catalog.load("kipu-quantum/iskay-quantum-optimizer")

## Exemplo 1: Função de custo simples

Considere a função de custo na formulação spin:
$$
C(x_0, x_1, x_2, x_3, x_4) = 1 + 1.5x_0 + 2x_1 + 1.3x_2 + 2.5x_0x_3 + 3.5x_1x_4 + 4x_0x_1x_2
$$

onde $(x_0, ..., x_4) \in {-1, 1}^5$.

A solução desta função de custo simples é
$$
(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)
$$
com valor mínimo $C^{*} = -6$

### 1. Criar a função objetivo

Começamos criando um dicionário com os coeficientes da função objetivo da seguinte forma:

In [ ]:
objective_func = {
    "()": 1,
    "(0,)": 1.5,
    "(1,)": 2,
    "(2,)": 1.3,
    "(0, 3)": 2.5,
    "(1, 4)": 3.5,
    "(0, 1, 2)": 4,
}

### 2. Executar o Optimizer
Resolvemos o problema executando o optimizer. Como $(x_0, ..., x_4) \in {-1, 1}^5$, devemos definir `problem_type=spin`.

In [ ]:
# Setup options to run the optimizer
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. Recuperar o resultado
A solução do problema de otimização é fornecida diretamente pelo optimizer.

In [ ]:
print(job.result())

Isso exibirá um dicionário da forma:

```
{'solution': {'0': -1, '1': -1, '2': -1, '3': 1, '4': 1},
 'solution_info': {'bitstring': '11100',
  'cost': -13.8,
  'seed_transpiler': 42,
  'mapping': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}},
 'prob_type': 'spin'}
```

Observe que o dicionário `solution` exibe o vetor resultado $(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)$.

## Exemplo 2: MaxCut

Muitos problemas de grafos como MaxCut ou conjunto independente máximo são problemas NP-difíceis e candidatos ideais para testar algoritmos quânticos e hardware. Este exemplo demonstra como resolver o problema MaxCut de um grafo 3-regular com o Quantum Optimizer.

Para executar este exemplo, você deve instalar o pacote `networkx` além do `qiskit-ibm-catalog`. Para instalá-lo, execute o seguinte comando:

In [ ]:
# %pip install networkx numpy

### 1. Criar a função objetivo
Comece gerando um grafo 3-regular aleatório. Para este grafo, definimos a função objetivo do problema MaxCut.

In [ ]:
import networkx as nx

# Create a random 3-regular graph
G = nx.random_regular_graph(3, 10, seed=42)


# Create the objective function for MaxCut in Ising formulation
def graph_to_ising_maxcut(G):
    """
    Convert a NetworkX graph to an Ising Hamiltonian for the Max-Cut problem.
    Args:
        G (networkx.Graph): The input graph.
    Returns:
        dict: The objective function of the Ising model
    """
    # Initialize the linear and quadratic coefficients
    objective_func = {}
    # Populate the coefficients
    for i, j in G.edges:
        objective_func[f"({i}, {j})"] = 0.5
    return objective_func


objective_func = graph_to_ising_maxcut(G)

### 2. Executar o Optimizer
Resolva o problema executando o optimizer.

In [ ]:
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. Recuperar o resultado
Recupere o resultado e mapeie a string de bits da solução de volta aos nós do grafo original.

In [ ]:
print(job.result())

A solução para o problema MaxCut está diretamente contida no sub-dicionário `solution` do objeto de resultado

In [ ]:
maxcut_solution = job.result()["solution"]

## Exemplo 3: Instâncias de benchmark
As instâncias de benchmark estão disponíveis no GitHub: [instâncias de benchmark da Kipu.](https://github.com/Kipu-Quantum-GmbH/benchmark-instances)

As instâncias podem ser carregadas usando a biblioteca `pygithub`. Para instalá-la, execute o seguinte comando:

In [ ]:
# %pip install pygithub

Os caminhos para as instâncias de benchmark são:

**Maxcut:**
- `'maxcut/maxcut_28_nodes.json'`
- `'maxcut/maxcut_30_nodes.json'`
- `'maxcut/maxcut_32_nodes.json'`
- `'maxcut/maxcut_80_nodes.json'`
- `'maxcut/maxcut_100_nodes.json'`
- `'maxcut/maxcut_120_nodes.json'`

**HUBO:**
- `'HUBO/hubo1_marrakesh.json'`
- `'HUBO/hubo2_marrakesh.json'`

Para reproduzir o desempenho do benchmark para as instâncias HUBO, selecione o backend `ibm_marrakesh` e defina `direct_qubit_mapping` como `True` no sub-dicionário `options`.
O exemplo a seguir executa a instância Maxcut com 32 nós.

In [ ]:
from github import Github
import urllib
import json
import ast

repo = "Kipu-Quantum-GmbH/benchmark-instances"
path = "maxcut/maxcut_32_nodes.json"
gh = Github()
repo = gh.get_repo(repo)
branch = "main"
file = repo.get_contents(urllib.parse.quote(path), ref=branch)

# load json file with benchmark problem
problem_json = json.loads(file.decoded_content)

# convert objective function to compatible format
objective_func = {
    key: ast.literal_eval(value) for key, value in problem_json.items()
}


# Setup configuration to run the optimizer
options = {
    "shots": 5_000,
    "num_iterations": 5,
    "use_session": True,
    "direct_qubit_mapping": False,
}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": "ibm_brisbane",
    "options": options,
}

job = optimizer.run(**arguments)

result = job.result()

## Casos de uso
Casos de uso típicos para o solver de otimização são problemas de otimização combinatória. Você pode resolver problemas de muitas indústrias como finanças, farmácia ou logística. Alguns exemplos:
* Otimização de portfólio (QUBO): [publicação científica](https://doi.org/10.1103/PhysRevApplied.22.054037) e [white paper](https://kipu-quantum.com/zope64/kipu_2024/content/e3915/e3916/e4187/White-Paper-2-Financial-modeling-on-quantum-computers-using-digitally-compressed-algorithms-1.pdf)
* Dobramento de proteínas (HUBO): [publicação científica](https://doi.org/10.1103/PhysRevApplied.20.014024)
* Agendamento logístico (QUBO): [publicação científica](https://doi.org/10.1103/PhysRevApplied.22.064068)
* Otimização de redes: [webinar](https://www.youtube.com/watch?v=w5SrCIK88No)
* Market split (QUBO): [tutorial](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer)

Se você tiver interesse em abordar um caso de uso específico e desenvolver um mapeamento dedicado, podemos ajudá-lo. [Entre em contato conosco.](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5)

## Obter suporte
Para suporte, entre em contato pelo e-mail support@kipu-quantum.com.

## Próximos passos
- [Solicite acesso ao Quantum Optimizer da Kipu Quantum](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5)
- Experimente o tutorial [Resolva o problema de Market Split com o Iskay Quantum Optimizer da Kipu Quantum](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer).
- Consulte [Romero, S. V., et al. (2025).  Bias-Field Digitized Counterdiabatic Quantum Algorithm for Higher-Order Binary Optimization. arXiv preprint arXiv:2409.04477.](https://arxiv.org/abs/2409.04477)
- Consulte [Cadavid, A. G., et al. (2024).  Bias-field digitized counterdiabatic quantum optimization. arXiv preprint arXiv:2405.13898.](https://arxiv.org/abs/2405.13898)
- Consulte [Chandarana, P., et al. (2025).  Runtime Quantum Advantage with Digital Quantum Optimization. arXiv preprint arXiv:2505.08663.](https://arxiv.org/abs/2505.08663)

## Informações adicionais
Iskay, assim como o nome da nossa empresa Kipu Quantum, é uma palavra peruana. Embora sejamos uma startup da Alemanha, essas palavras vêm do país natal de um dos nossos co-fundadores, onde o Quipu foi uma das primeiras máquinas de cálculo desenvolvidas pela humanidade há 2000 anos a.C.